# 0.Library

In [1]:
import pandas as pd
from pathlib import Path
import re

import importlib
import sys
sys.path.append('../') 
from features import data_utils as du
from features import data_pipeline as dp
from features import general_func as gf
import constants_data as cd

# Reload
importlib.reload(du)
importlib.reload(dp)
importlib.reload(gf)

<module 'features.general_func' from 'd:\\my-github\\detection_AD_with_VR_data\\src\\notebooks\\..\\features\\general_func.py'>

# 1. Paths and Constants

In [ ]:
# Constants and paths
parent_folder = Path("../..") # go 2 folder up= "../.."
data_folder   = parent_folder / "data"
input_path_of_json   = data_folder / "patients_data_log.json"
patients_data_log_df = pd.read_json(input_path_of_json)

patient_ids          = patients_data_log_df["PatientID"].to_list()
patients_data_folder = data_folder / "PatientsData"

patient_dict = cd.patient_dict
phase_name_list = cd.phase_name_list
event_empty_dictionaries = [cd.reading_time_dict, cd.hover_dict, cd.press_dict, cd.grab_dict, cd.gaze_dict, cd.behavior_dict, cd.temporal_dict]
extra_features_empty_dictionaries = [cd.obj_recognition_dict, cd.visuospatial_dict, cd.memory_dict]
trajectory_empty_dictionaries = [cd.headset_dict, cd.controller_dict]

all_four_phase_dict = []

# 2. Creating extracted_features_df columns

In [37]:
basic_event_col = []
for value in event_empty_dictionaries:
    column = value.keys()
    basic_event_col += list(column)

trajectory_col = []
for value in trajectory_empty_dictionaries:
    column = value.keys()
    trajectory_col += list(column)

In [38]:
all_phase_column_lists = []
all_phase_column_lists.append(list(patient_dict.keys())) 

for phase in phase_name_list:
    phase_column_list = []
    
    for value in basic_event_col:
        
        phase_column = phase + '_' + value 
        phase_column_list.append(phase_column.lower())

    if phase == phase_name_list[1]: # ObjectRecognition
        phase_column_list.extend(extra_features_empty_dictionaries[0].keys())

    elif phase == phase_name_list[2]: # Visuospatial
        phase_column_list.extend(extra_features_empty_dictionaries[1].keys())

    elif phase == phase_name_list[3]: # Memory
        phase_column_list.extend(extra_features_empty_dictionaries[2].keys())
    
    # add trajectory
    for value in trajectory_col:
        
        phase_column = phase + '_' + value 
        phase_column_list.append(phase_column.lower())
    
    all_phase_column_lists.append(phase_column_list)

all_phase_column_lists

[['Paitent_id',
  'Age',
  'Gender',
  'dominant_hand',
  'Sessions_Completed_out_of_4',
  'Help_Rating_out_of_5'],
 ['tutorial_total_reading_time',
  'tutorial_mean_reading_time',
  'tutorial_max_reading_time',
  'tutorial_median_reading_time',
  'tutorial_std_reading_time',
  'tutorial_intensity_reading_time',
  'tutorial_total_count_hover',
  'tutorial_total_duration_hover',
  'tutorial_mean_duration_hover',
  'tutorial_max_duration_hover',
  'tutorial_median_duration_hover',
  'tutorial_std_duration_hover',
  'tutorial_intensity_hover',
  'tutorial_total_count_press',
  'tutorial_total_duration_press',
  'tutorial_mean_duration_press',
  'tutorial_max_duration_press',
  'tutorial_median_duration_press',
  'tutorial_std_duration_press',
  'tutorial_intensity_press',
  'tutorial_total_count_grab',
  'tutorial_total_duration_grab',
  'tutorial_mean_duration_grab',
  'tutorial_max_duration_grab',
  'tutorial_median_duration_grab',
  'tutorial_std_duration_grab',
  'tutorial_intensity_g

# 3. Creating df

In [52]:
columns = [col for lst in all_phase_column_lists for col in lst]
extracted_features_df = pd.DataFrame(columns=columns)

extracted_features_df.columns

Index(['Paitent_id', 'Age', 'Gender', 'dominant_hand',
       'Sessions_Completed_out_of_4', 'Help_Rating_out_of_5',
       'tutorial_total_reading_time', 'tutorial_mean_reading_time',
       'tutorial_max_reading_time', 'tutorial_median_reading_time',
       ...
       'memory_not_dominant_hand_y_range', 'memory_not_dominant_hand_z_range',
       'memory_dominant_hand_trigger_press_count',
       'memory_not_dominant_hand_trigger_press_count',
       'memory_dominant_hand_trigger_pressure_mean',
       'memory_not_dominant_hand_trigger_pressure_mean',
       'memory_dominant_hand_grip_count',
       'memory_not_dominant_hand_grip_count', 'memory_dominant_hand_grip_mean',
       'memory_not_dominant_hand_grip_mean'],
      dtype='object', length=356)

# 4. Filling the extracted_features_df

In [ ]:
print("STARTING....")
# First loop
for patient_id in patient_ids:
    
    print("============================================================")
    # First loop : fill patient data
    print(f"Extracting row for patient : {patient_id}")
    
    # extract data path
    patient_folder_path = du.find_patient_folder(patients_data_folder=patients_data_folder,
                                                 patient_id=patient_id)
    row_index = patient_ids.index(patient_id)

    # filling patient dictionary
    patient_dict = du.filling_patient_dict(patients_data_log_df, patient_dict, row_index)
    
    # filling phase dictionaries
    path_to_check  = patient_folder_path / "clean_data"
    csv_files_list = du.find_csv_file(patient_folder_path / "clean_data")

    # set dominant and not dominant hand
    dominant_hand = patient_dict["dominant_hand"]
    not_dominant_hand = 'Left' if dominant_hand == 'Right' else 'Right'

    print("Patient infos extracted successfully.")

    # phase id (range 1 to 4)
    phase_id = 1

    # Second loop: now fill 4 phases (event+trajectory)
    for i in range(0,8,2):
        print(f"starting phase : {i}")

        # Event file
        csv_file_event = csv_files_list[i]
        df_event       = pd.read_csv(path_to_check / csv_file_event)

        # extract phase duration
        phase_duration = du.extract_phase_duration(df_event)

        # extract phase name
        match      = re.search(r'_([^_]+)_events\.csv$', csv_file_event)
        phase_name =  match.group(1) if match else sys.exit("phase name not found!!!")

        # filling
        phase_name = phase_name if phase_name in phase_name_list else gf.fail(msg="phase_name is found but it's invalid!!!", error="Key not found")

        # filling for events
        event_dict = dp.creating_event_dictionary(df_event, phase_name , event_empty_dictionaries, extra_features_empty_dictionaries, phase_duration)
        
        # Trajectory file
        csv_file_trajectory = csv_files_list[i+1]
        df_trajectory       = pd.read_csv(path_to_check / csv_file_trajectory)

        # filling for trajectory
        trajectory_dict = dp.creating_trajectory_dictionary(df_trajectory, dominant_hand, not_dominant_hand, trajectory_empty_dictionaries)
        
        # prepareing event_dict and trajectory_dict columns for assgining (with extra feautres, depends on phase_name)
        if  phase_name == phase_name_list[0]: # Tutorial
            # add prefix tutorial_ to all keys
            event_dict      = {f"tutorial_{k}": v for k, v in event_dict.items()}
            trajectory_dict = {f"tutorial_{k}": v for k, v in trajectory_dict.items()}

        elif phase_name == phase_name_list[1]: # ObjectRecognition
            
            # extract list of extra features(they already have the prefix!)
            extra_features_keys = list(extra_features_empty_dictionaries[0].keys())

            # add prefix objectrecognition_ to all keys
            event_dict = {
                (f"objectrecognition_{k}" if k not in extra_features_keys else k): v
                for k, v in event_dict.items()
            }

            trajectory_dict = {f"objectrecognition_{k}": v for k, v in trajectory_dict.items()}
            
        elif phase_name == phase_name_list[2]: # Visuospatial
            
            # extract list of extra features(they already have the prefix!)
            extra_features_keys = list(extra_features_empty_dictionaries[1].keys())

            # add prefix visuospatial_ to all keys
            event_dict = {
                (f"visuospatial_{k}" if k not in extra_features_keys else k): v
                for k, v in event_dict.items()
            }

            trajectory_dict = {f"visuospatial_{k}": v for k, v in trajectory_dict.items()}

        elif phase_name == phase_name_list[3]: # Memory
            
            # extract list of extra features(they already have the prefix!)
            extra_features_keys = list(extra_features_empty_dictionaries[2].keys())

            # add prefix memory_ to all keys
            event_dict = {
                (f"memory_{k}" if k not in extra_features_keys else k): v
                for k, v in event_dict.items()
            }

            trajectory_dict = {f"memory_{k}": v for k, v in trajectory_dict.items()}
        
        concated_phase_dict = event_dict | trajectory_dict
        all_four_phase_dict.append(concated_phase_dict)

        print(f"Features extracted successfully.")

    print("All four phases are done.")

    # concate phase dictionary and patient
    single_row_of_df = patient_dict | all_four_phase_dict[0] | all_four_phase_dict[1] | all_four_phase_dict[2] | all_four_phase_dict[3]

    # add single row to final dataframe
    extracted_features_df.loc[len(extracted_features_df)] = single_row_of_df
    print("New row added to df successfully.")
    
    print("============================================================")
    
print("Main process is finished.")

STARTING....


NameError: name 'patient_ids' is not defined

In [54]:
extracted_features_df

,Paitent_id,Age,Gender,dominant_hand,Sessions_Completed_out_of_4,Help_Rating_out_of_5,tutorial_total_reading_time,tutorial_mean_reading_time,tutorial_max_reading_time,tutorial_median_reading_time,...,memory_not_dominant_hand_y_range,memory_not_dominant_hand_z_range,memory_dominant_hand_trigger_press_count,memory_not_dominant_hand_trigger_press_count,memory_dominant_hand_trigger_pressure_mean,memory_not_dominant_hand_trigger_pressure_mean,memory_dominant_hand_grip_count,memory_not_dominant_hand_grip_count,memory_dominant_hand_grip_mean,memory_not_dominant_hand_grip_mean
0,P001,None,Female,Right,4,1,45.03,15.01,21.0,17.23,...,0.8,0.5,35,0,0.01,0.0,1275,0,0.4,0.0
1,P002,59,Female,Right,4,3,45.03,15.01,21.0,17.23,...,0.8,0.5,35,0,0.01,0.0,1275,0,0.4,0.0
2,P003,82,Male,Right,4,4,45.03,15.01,21.0,17.23,...,0.8,0.5,35,0,0.01,0.0,1275,0,0.4,0.0
3,P004,75,Male,Left,4,3,45.03,15.01,21.0,17.23,...,0.8,0.5,35,0,0.01,0.0,1275,0,0.4,0.0
4,P005,62,Male,Right,4,4,45.03,15.01,21.0,17.23,...,0.8,0.5,35,0,0.01,0.0,1275,0,0.4,0.0
5,P006,78,Female,Right,4,3,45.03,15.01,21.0,17.23,...,0.8,0.5,35,0,0.01,0.0,1275,0,0.4,0.0
6,P007,71,Female,Right,4,3,45.03,15.01,21.0,17.23,...,0.8,0.5,35,0,0.01,0.0,1275,0,0.4,0.0
7,P008,69,Female,Right,4,4,45.03,15.01,21.0,17.23,...,0.8,0.5,35,0,0.01,0.0,1275,0,0.4,0.0
8,P009,73,Male,Right,4,2,45.03,15.01,21.0,17.23,...,0.8,0.5,35,0,0.01,0.0,1275,0,0.4,0.0
9,P010,81,Female,Right,4,3,45.03,15.01,21.0,17.23,...,0.8,0.5,35,0,0.01,0.0,1275,0,0.4,0.0


# 5. Save df

In [58]:
path_df = data_folder / "produced_csv"
name = 'extracted_features.csv'
extracted_features_df.to_csv(path_df / name)